# Detecção de Desgaste em Pneus com CNNs — TCC
**Gabriel Gomes Galikosky** · IFC Campus Araquari · 2026

Comparação de três arquiteturas (Baseline, VGG16, ResNet50) para classificar
a condição de pneus em `good`, `worn` e `cracked`.

> Em **Ambiente de execução → Alterar tipo de hardware**, selecione **GPU (T4)**.


## 1. Setup — clonar/enviar o projeto e instalar dependências


In [ ]:
# Se o projeto estiver no GitHub, clone-o; senão, faça upload da pasta src/ e scripts/.
# !git clone https://github.com/<seu-usuario>/tcc-cnn.git && cd tcc-cnn
!pip -q install opencv-python-headless split-folders


## 2. Dados
Escolha **A** (datasets reais do Kaggle) ou **B** (sintéticos, só para testar).


In [ ]:
# --- Opção A: Kaggle (requer kaggle.json) ---
# from google.colab import files; files.upload()  # envie kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !python scripts/download_data.py --dest data/raw

# --- Opção B: dados sintéticos (smoke test) ---
!python scripts/make_synthetic_data.py --dest data/raw --per-class 60


## 3. Consolidação + divisão estratificada (70/15/15, seed=42)


In [ ]:
from src import data_setup
from src.config import default_config, RANDOM_SEED
cfg = default_config()
print(data_setup.consolidate(cfg.raw_dir, cfg.consolidated_dir))
print(data_setup.split_dataset(cfg.consolidated_dir, cfg.split_dir,
      ratios=(cfg.train_ratio, cfg.val_ratio, cfg.test_ratio), seed=RANDOM_SEED))


## 4. Visualizar efeito do CLAHE (experimento auxiliar)


In [ ]:
import glob
from src.preprocessing import preview_clahe
sample = glob.glob('data/consolidated/good/*')[0]
preview_clahe(sample, cfg, 'outputs/clahe_preview.png')
from IPython.display import Image; Image('outputs/clahe_preview.png')


## 5. Executar o experimento completo
Os 3 modelos, com e sem CLAHE. Em GPU, use as 50 épocas padrão (sem `--quick`).


In [ ]:
!python -m scripts.run_experiment --models baseline vgg16 resnet50 --clahe both


## 6. Resultados
Tabela comparativa, relatório e gráficos em `outputs/`.


In [ ]:
from IPython.display import Markdown
Markdown(open('outputs/RELATORIO.md', encoding='utf-8').read())


In [ ]:
from IPython.display import Image
for m in ['baseline','vgg16','resnet50']:
    display(Image(f'outputs/{m}/confusion_matrix.png'))
    display(Image(f'outputs/{m}/learning_curves.png'))
